# Which Investors Produce More Informative Asset Embeddings?

## Active Share, turnover, and the economic content of mutual-fund ownership

This notebook summarizes the active-versus-passive extension of the 2005Q1–2024Q4 asset-embedding replication. It is organized as the research developed: beginning with a simple active/passive question, confronting measurement and identification problems, progressively adding fair comparisons and robustness tests, and ending with a more precise economic mechanism.

The central conclusion is not that active managers are better investors. It is that **market-distinctive ownership decisions are more informative observations for learning relationships among assets**, especially when the representation is highly compressed.

## 1. Starting question and final answer

The extension began with the professor's question:

> Do embeddings constructed from active mutual funds' holdings perform differently from embeddings constructed from passive mutual funds' holdings?

The short answer is **yes, but only in a structured way**. High-Active-Share funds generate substantially more effective low-dimensional Binary embeddings than passive-like funds. Turnover alone contributes little. When more dimensions are available, pooling the complete S12 universe becomes more effective. The advantage also disappears or reverses when embeddings use holding sizes rather than ownership incidence.

That pattern led to the sharper conclusion:

> **Portfolio distinctiveness, rather than trading frequency, identifies the investors whose discretionary ownership choices generate more valuation-informative asset embeddings.**

## 2. Load the completed local evidence

This notebook performs no WRDS queries and refits no embeddings. It reads the immutable, checkpointed outputs created by `Active_Passive_Extension.ipynb`.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != 'Asset_Embeddings':
    candidate = PROJECT_ROOT / 'Projects' / 'Asset_Embeddings'
    if candidate.exists():
        PROJECT_ROOT = candidate.resolve()
if PROJECT_ROOT.name != 'Asset_Embeddings':
    raise RuntimeError('Open this notebook from the Asset_Embeddings project directory.')
os.chdir(PROJECT_ROOT)

EXTENSION_ROOT = PROJECT_ROOT / 'active_passive_extension'
FIGURE_ROOT = EXTENSION_ROOT / 'figures' / 'summary'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

active_share = pd.read_parquet(EXTENSION_ROOT / 'measures' / 's12_active_share_80q.parquet')
turnover = pd.read_parquet(EXTENSION_ROOT / 'measures' / 's12_return_adjusted_turnover_80q_v2_total_return.parquet')
master = pd.read_parquet(EXTENSION_ROOT / 'measures' / 's12_activeness_master_80q.parquet')
main_results = pd.read_parquet(EXTENSION_ROOT / 'benchmarks' / 'all_s12_control' / 'extension_results_with_all_s12.parquet')
main_summary = pd.read_parquet(EXTENSION_ROOT / 'benchmarks' / 'all_s12_control' / 'extension_summary_with_all_s12.parquet')
matched_summary = pd.read_parquet(EXTENSION_ROOT / 'benchmarks' / 'equal_fund_count_binary' / 'summary.parquet')
paired_inference = pd.read_parquet(EXTENSION_ROOT / 'benchmarks' / 'statistical_inference' / 'paired_inference.parquet')
factorial = pd.read_parquet(EXTENSION_ROOT / 'benchmarks' / 'factorial_effects' / 'block_bootstrap_inference.parquet')
crisis = pd.read_parquet(EXTENSION_ROOT / 'benchmarks' / 'crisis_performance' / 's12_crisis_summary.parquet')
dimension_curves = pd.read_parquet(EXTENSION_ROOT / 'benchmarks' / 'binary_dimension_curves' / 'dimension_summary.parquet')

required = {
    'Active Share fund-quarters': len(active_share),
    'Valid turnover fund-quarters': int(turnover['turnover_valid'].sum()),
    'Main benchmark rows': len(main_results),
    'Matched benchmark rows': int(matched_summary['seed_quarters'].sum()),
    'Dimension-curve rows': len(dimension_curves),
}
display(pd.Series(required, name='observations').rename_axis('artifact').reset_index())

COLORS = {
    'all_s12':'#244A73', 'passive_like':'#AEB4BE',
    'stable_stock_picker':'#4F86B5', 'benchmark_rebalancer':'#D39A3A',
    'tactical_active':'#B64B4B', 'combined_s12_s34':'#5B6573',
}
LABELS = {
    'all_s12':'All S12', 'passive_like':'Passive-like',
    'stable_stock_picker':'Stable stock picker', 'benchmark_rebalancer':'Benchmark rebalancer',
    'tactical_active':'Tactical active', 'combined_s12_s34':'Combined S12–S34',
}
plt.rcParams.update({'figure.dpi':120, 'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.grid':True, 'grid.alpha':0.20, 'font.size':10})

## 3. Why a reported active/passive label was insufficient

S12 identifies individual mutual funds, so it became the classification universe. S34 remains useful in the original combined replication, but an S34 manager can operate many funds with different strategies and cannot inherit a single fund-level active/passive label.

CRSP's historical index flag was used as external validation where a defensible ticker linkage existed. MFLINKS was not available under the WRDS subscription, limiting reported-status coverage to approximately 43%. More importantly, reported labels do not reveal how distinctive or how actively traded a portfolio actually is.

We therefore constructed two independent holdings-based measures: Active Share and return-adjusted turnover.

## 4. Measuring portfolio distinctiveness and trading

### Active Share

For fund $i$ in quarter $t$, relative to the CRSP common-equity market portfolio:

$$AS_{it}=\frac{1}{2}\sum_a |w_{iat}-w^{mkt}_{at}|.$$

Active Share captures **what the fund chooses to own differently from the market**.

### Return-adjusted turnover

The preceding portfolio is first carried forward using realized firm returns:

$$w^{BH}_{ia,t}=\frac{w_{ia,t-1}(1+R_{a,t})}{\sum_b w_{ib,t-1}(1+R_{b,t})},$$

and turnover is:

$$TO_{it}=\frac{1}{2}\sum_a|w_{ia,t}-w^{BH}_{ia,t}|.$$

This removes weight changes caused only by asset returns. Turnover was decomposed exactly into entries/exits—the extensive margin—and reweighting of continuing positions—the intensive margin.

In [ ]:
measure_summary = pd.DataFrame({
    'Measure':['Active Share','Return-adjusted turnover','Extensive turnover','Intensive turnover'],
    'Observations':[len(active_share), int(turnover['turnover_valid'].sum()),
                    int(turnover['turnover_valid'].sum()), int(turnover['turnover_valid'].sum())],
    'Mean':[active_share['active_share'].mean(),
            turnover.loc[turnover['turnover_valid'],'return_adjusted_turnover'].mean(),
            turnover.loc[turnover['turnover_valid'],'extensive_turnover'].mean(),
            turnover.loc[turnover['turnover_valid'],'intensive_turnover'].mean()],
    'Median':[active_share['active_share'].median(),
              turnover.loc[turnover['turnover_valid'],'return_adjusted_turnover'].median(),
              turnover.loc[turnover['turnover_valid'],'extensive_turnover'].median(),
              turnover.loc[turnover['turnover_valid'],'intensive_turnover'].median()],
})
display(measure_summary.style.format({'Mean':'{:.1%}','Median':'{:.1%}','Observations':'{:,.0f}'}))
print('Active Share–turnover correlation:',
      f"{master.loc[master['turnover_valid'].fillna(False), ['active_share','return_adjusted_turnover']].corr().iloc[0,1]:.3f}")

## 5. Moving from a binary label to a 2×2 behavioral design

Active Share and turnover were only modestly correlated, so they describe different forms of activeness. Quarterly median splits produced four groups:

| Group | Active Share | Turnover | Economic interpretation |
|---|---|---|---|
| Passive-like | Low | Low | Market-like and trades little |
| Stable stock picker | High | Low | Distinctive choices held persistently |
| Benchmark rebalancer | Low | High | Market-like but trades frequently |
| Tactical active | High | High | Distinctive choices and substantial trading |

This design allowed us to separate **portfolio distinctiveness** from **trading intensity**, rather than treating every nominally active fund as equivalent.

## 6. Making the embedding comparison fair

The original replication combined S12 and S34. The extension's primary comparison is entirely within S12; combined S12–S34 is only an external reference.

To prevent different asset universes from driving performance:

- firms had to be represented in every behavioral group;
- the same quarterly firm mapping was used for all group embeddings;
- funds retained at least ten common-universe positions;
- relative valuation was rebuilt fresh on the common sample;
- the original train/test assignment was preserved; and
- scaling and ridge selection used training firms only.

We estimated RS-Binary and RS-L-Min embeddings at 4D and 10D over 79 quarters. An all-S12 control was then fitted on the identical common firm universe.

## 7. First result: the active advantage is model- and dimension-specific

The first benchmark showed a striking but structured pattern. Tactical-active and stable stock-picking funds were strongest for compressed Binary embeddings. At ten dimensions, pooling all S12 funds became superior. The effect did not generalize to L-Min, which incorporates holding sizes.

In [ ]:
groups = ['all_s12','passive_like','stable_stock_picker','benchmark_rebalancer','tactical_active']
plot = main_summary[main_summary['investor_group'].isin(groups)].copy()
plot['specification'] = plot['embedding_model'].str.replace('rs_','',regex=False).str.upper() + ' ' + plot['dimension'].astype(str) + 'D'
specs = ['BINARY 4D','BINARY 10D','L_MIN 4D','L_MIN 10D']
fig, axes = plt.subplots(1,4,figsize=(19,5.5),sharey=True)
for axis, spec in zip(axes,specs):
    data = plot[plot['specification'].eq(spec)].set_index('investor_group').reindex(groups)
    values = data['mean_test_variance_r2'].to_numpy()
    bars = axis.bar(np.arange(len(groups)), values, color=[COLORS[g] for g in groups], width=.72)
    axis.set_title(spec.replace('_','-'),loc='left',fontweight='semibold')
    axis.set_xticks(np.arange(len(groups)),[LABELS[g] for g in groups],rotation=55,ha='right')
    axis.set_ylim(0,.70)
    axis.grid(axis='y')
    for bar,value in zip(bars,values):
        axis.text(bar.get_x()+bar.get_width()/2,value+.012,f'{value:.1%}',ha='center',fontsize=9)
axes[0].set_ylabel('Mean out-of-sample valuation $R^2$')
fig.suptitle('Behavioral Fund Groups Produce Different Asset-Embedding Information',fontweight='bold',fontsize=14)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'main_behavioral_benchmark.png',dpi=300,bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'main_behavioral_benchmark.pdf',bbox_inches='tight')
plt.show()

### What this changed in our interpretation

At Binary 4D, tactical-active reached 44.1%, compared with 30.6% for passive-like and 37.8% for all S12. At Binary 10D, all S12 reached 61.3% and overtook every subgroup. L-Min favored pooling rather than active selection.

This suggested that active ownership might supply concentrated information about **who chooses to own which stocks**, while the broader universe contains additional useful information that requires more dimensions to represent. But subgroup sizes still offered an obvious alternative explanation.

## 8. Removing the subgroup-size explanation

Every behavioral group was downsampled to the same number of funds within each quarter, repeated across five deterministic seeds. The result barely changed.

In [ ]:
matched_groups = ['passive_like','benchmark_rebalancer','stable_stock_picker','tactical_active']
fig, axes = plt.subplots(1,2,figsize=(12.5,5.2),sharey=True)
for axis,dimension in zip(axes,[4,10]):
    data = matched_summary[matched_summary['dimension'].eq(dimension)].set_index('investor_group').reindex(matched_groups)
    values = data['mean_test_variance_r2'].to_numpy()
    bars = axis.bar(np.arange(4),values,color=[COLORS[g] for g in matched_groups],width=.68)
    axis.set_title(f'RS-Binary {dimension}D',loc='left',fontweight='semibold')
    axis.set_xticks(np.arange(4),[LABELS[g] for g in matched_groups],rotation=35,ha='right')
    axis.set_ylim(0,.60); axis.grid(axis='y')
    for bar,value in zip(bars,values):
        axis.text(bar.get_x()+bar.get_width()/2,value+.012,f'{value:.1%}',ha='center',fontsize=10)
axes[0].set_ylabel('Mean out-of-sample valuation $R^2$')
fig.suptitle('Equal Fund Counts Leave the Behavioral Ranking Intact',fontweight='bold',fontsize=14)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'equal_fund_count.png',dpi=300,bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'equal_fund_count.pdf',bbox_inches='tight')
plt.show()
display(matched_summary.sort_values(['dimension','mean_test_variance_r2'],ascending=[True,False]).style.format({
    'mean_test_variance_r2':'{:.1%}','standard_deviation_r2':'{:.1%}',
    'minimum_matched_funds':'{:,.0f}','median_matched_funds':'{:,.0f}'}))

## 9. The 2×2 design revealed what actually matters

Once sample size was equalized, we formally decomposed the results into an Active Share main effect, turnover main effect, and interaction. The five seeds were averaged within quarter before four-quarter block-bootstrap inference, leaving 79 temporal observations.

In [ ]:
effect_labels = {'active_share_main_effect':'Active Share','turnover_main_effect':'Turnover',
                 'active_share_x_turnover':'Interaction'}
effect_colors = {'active_share_main_effect':'#4F86B5','turnover_main_effect':'#AEB4BE',
                 'active_share_x_turnover':'#D39A3A'}
fig,axes=plt.subplots(1,2,figsize=(12,4.8),sharey=True)
for axis,dimension in zip(axes,[4,10]):
    data=factorial[factorial['dimension'].eq(dimension)].copy()
    positions=np.arange(len(data)); values=data['mean_r2_effect'].to_numpy()
    errors=np.vstack([values-data['ci_2_5'].to_numpy(),data['ci_97_5'].to_numpy()-values])
    bars=axis.bar(positions,values,color=[effect_colors[e] for e in data['effect']],width=.65)
    axis.errorbar(positions,values,yerr=errors,fmt='none',ecolor='#333333',capsize=4,linewidth=1.2)
    axis.axhline(0,color='#333333',linewidth=.8)
    axis.set_xticks(positions,[effect_labels[e] for e in data['effect']],rotation=20,ha='right')
    axis.set_title(f'RS-Binary {dimension}D',loc='left',fontweight='semibold')
    for bar,value in zip(bars,values):
        axis.text(bar.get_x()+bar.get_width()/2,value+.006,f'{value:+.1%}',ha='center',fontsize=10)
axes[0].set_ylabel('Effect on out-of-sample valuation $R^2$')
fig.suptitle('Portfolio Distinctiveness Matters More Than Trading Frequency',fontweight='bold',fontsize=14)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'factorial_effects.png',dpi=300,bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'factorial_effects.pdf',bbox_inches='tight')
plt.show()
display(factorial.assign(effect=factorial['effect'].map(effect_labels)).style.format({
    'mean_r2_effect':'{:+.1%}','ci_2_5':'{:+.1%}','ci_97_5':'{:+.1%}','p_value':'{:.4f}'}))

The Active Share effect is large and significant: +11.0 percentage points at 4D and +7.6 points at 10D. Turnover is small and statistically indistinguishable from zero. The 10D interaction is positive, indicating that turnover becomes useful only when combined with distinctive portfolios.

This shifted the project away from a generic active/passive story:

> **What a fund chooses to own differently from the market is more informative than how frequently it changes those choices.**

## 10. Stress periods: GFC and COVID-19

We next asked whether the behavioral ranking changed during market disruption. The GFC window contains eight quarters, 2007Q3–2009Q2. The acute COVID window contains four quarters, 2020Q1–2020Q4.

During the GFC, stable and tactical funds each beat passive-like funds by approximately 10 points in Binary 4D, with exact paired p-values of .047 and .031. During COVID, tactical-active exceeded passive-like by 25.2 points at Binary 4D and 15.9 points at Binary 10D. Those COVID estimates are economically large and directionally consistent, but four quarters cannot deliver conventional two-sided exact significance.

In [ ]:
crisis_groups=['all_s12','passive_like','stable_stock_picker','tactical_active']
fig,axes=plt.subplots(1,2,figsize=(13,5),sharey=True)
for axis,(regime,title) in zip(axes,[('GFC_2007Q3_2009Q2','Global Financial Crisis'),
                                      ('COVID_2020Q1_2020Q4','COVID-19 disruption')]):
    data=crisis[(crisis['market_regime'].eq(regime)) & (crisis['embedding_model'].eq('rs_binary'))
                & (crisis['dimension'].eq(4))].set_index('investor_group').reindex(crisis_groups)
    values=data['mean_test_variance_r2'].to_numpy()
    bars=axis.bar(np.arange(len(crisis_groups)),values,color=[COLORS[g] for g in crisis_groups],width=.68)
    axis.set_xticks(np.arange(len(crisis_groups)),[LABELS[g] for g in crisis_groups],rotation=30,ha='right')
    axis.set_title(title,loc='left',fontweight='semibold'); axis.set_ylim(0,.68); axis.grid(axis='y')
    for bar,value in zip(bars,values):
        axis.text(bar.get_x()+bar.get_width()/2,value+.012,f'{value:.1%}',ha='center',fontsize=10)
axes[0].set_ylabel('Mean Binary 4D valuation $R^2$')
fig.suptitle('Market-Distinctive Ownership Remained Informative During Stress',fontweight='bold',fontsize=14)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'crisis_binary_4d.png',dpi=300,bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'crisis_binary_4d.pdf',bbox_inches='tight')
plt.show()

## 11. The spectral puzzle changed the proposed mechanism

A natural initial explanation was that active holdings might form a cleaner, more compressible low-rank matrix. The spectral evidence rejected that story. Passive-like holdings are much easier to reconstruct: their first four Binary components explain 58.2% of holdings variance, compared with only 22.2% for tactical-active funds.

Yet tactical-active components predict valuation much better at low dimension. This separates **statistical importance in the holdings matrix** from **economic importance for the downstream task**. PCA is unsupervised: explaining widespread benchmark ownership is not the same as learning economically useful distinctions among firms.

In [ ]:
curve_groups=['all_s12','passive_like','stable_stock_picker','benchmark_rebalancer','tactical_active']
fig,axes=plt.subplots(1,2,figsize=(15,5.5))
for group in curve_groups:
    data=dimension_curves[dimension_curves['investor_group'].eq(group)].sort_values('dimension')
    axes[0].plot(data['dimension'],data['mean_cumulative_variance'],label=LABELS[group],
                 color=COLORS[group],linewidth=2.2)
    axes[1].plot(data['dimension'],data['mean_test_variance_r2'],label=LABELS[group],
                 color=COLORS[group],linewidth=2.2)
axes[0].set_title('A. Holdings variance explained',loc='left',fontweight='semibold')
axes[1].set_title('B. Economic information',loc='left',fontweight='semibold')
axes[0].set_ylabel('Mean cumulative explained variance')
axes[1].set_ylabel('Mean out-of-sample valuation $R^2$')
for axis in axes:
    axis.set_xlabel('Embedding dimensions'); axis.set_xticks([1,2,3,4,5,10,15,20]); axis.grid(True)
axes[1].legend(frameon=False,fontsize=9,loc='lower right')
fig.suptitle('Variance Explained Is Not the Same as Economic Information',fontweight='bold',fontsize=14)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'spectral_vs_predictive_curves.png',dpi=300,bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'spectral_vs_predictive_curves.pdf',bbox_inches='tight')
plt.show()

## 12. Where the useful information arrives

The dimension curve completed the mechanism:

- Stable stock pickers are strongest at 1D.
- Tactical-active becomes dominant at 3D–4D, receiving an unusually large gain at the third component.
- All S12 overtakes tactical-active at approximately 5D.
- All S12 continues accumulating useful information through dimensions 5–20.

At 4D, tactical-active produces 44.1% valuation $R^2$ while explaining only 22.2% of holdings variance. Passive-like produces only 30.6% valuation $R^2$ despite explaining 58.2% of holdings variance. At 20D, all S12 reaches 65.6%, demonstrating that the pooled universe contains more total useful information distributed across additional dimensions.

In [ ]:
selected_dimensions=[1,2,3,4,5,10,20]
final_curve_table=(dimension_curves[dimension_curves['dimension'].isin(selected_dimensions)]
    .pivot(index='dimension',columns='investor_group',values='mean_test_variance_r2')
    .reindex(columns=curve_groups))
display(final_curve_table.rename(columns=LABELS).style.format('{:.1%}').highlight_max(axis=1,color='#DCE9F5'))
incremental=(dimension_curves[dimension_curves['dimension'].le(10)]
    .pivot(index='dimension',columns='investor_group',values='mean_incremental_r2')
    .reindex(columns=curve_groups))
display(incremental.rename(columns=LABELS).style.format('{:+.1%}').background_gradient(cmap='Blues',axis=None))

## 13. Final conclusion

The project began by asking whether active and passive funds produce embeddings of different effectiveness. The evidence ultimately supports a more precise answer:

> **Holdings-based asset embeddings constructed from market-distinctive mutual funds significantly outperform passive-like fund embeddings when similarity is learned from low-dimensional binary ownership patterns. The advantage survives equal-fund-count sampling and serial-dependence-adjusted inference. It is driven primarily by Active Share rather than turnover and can exceed embeddings constructed from the entire S12 mutual-fund universe at four dimensions. The advantage diminishes once additional dimensions are available and does not extend to holdings-size embeddings.**

The findings are consistent with an economic interpretation based on discretionary revealed preference. A passive investor may jointly own two firms because both belong to its benchmark. A high-Active-Share investor had meaningful discretion not to own either firm; its decision to hold both is therefore a more informative signal about their latent relationship. Trading frequently does not create that information by itself.

The spectral results add an important distinction:

> **Passive holdings are easier to reconstruct; active holdings are more useful for low-dimensional representation learning.**

This is evidence about the information content of ownership choices—not evidence that active funds earn superior investment returns.

## 14. Limitations and reproducibility

Important limitations remain:

- Active Share uses the broad CRSP market rather than a fund-specific style benchmark, so specialized funds can appear highly active. Quarterly ranks mitigate, but do not eliminate, this issue.
- Reported-index validation is incomplete because MFLINKS was not accessible under the WRDS subscription.
- COVID inference is intrinsically low-powered with four quarterly observations.
- The embeddings are transductive: the unsupervised holdings decomposition uses the quarterly asset universe before the downstream valuation train/test regression. This follows the replication design but should be stated clearly.
- The mechanism is supported by predictive alignment and factorial contrasts; it does not establish that active managers possess superior private information or earn superior returns.

Reproducibility map:

- Construction and analysis: `Active_Passive_Extension.ipynb`
- Completed local artifacts: `active_passive_extension/`
- Summary figures: `active_passive_extension/figures/summary/`
- This narrative notebook: `Active_Passive_Extension_Summary.ipynb`